[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mongkol-saensuk/thai-ecommerce-reviews-sghs/blob/main/notebooks/SG_HS_MethodsX_experiments.ipynb)
&nbsp; ข้อมูลและโค้ด: https://github.com/mongkol-saensuk/thai-ecommerce-reviews-sghs

# Skip-gram + Hierarchical Softmax (SG-HS) สำหรับจำแนกความคิดเห็นลูกค้าไทยบน e-commerce

Notebook นี้รันการทดลองทั้งหมดตามต้นฉบับ MethodsX แบบเป็นขั้นตอน (Steps 3.2–6) และเก็บผลไว้ใน Google Drive

| ส่วน | ทำอะไร |
|:--|:--|
| 0 | ติดตั้งไลบรารี, ตั้งค่าการทดลองทั้งหมดในเซลล์เดียว |
| 1 | โหลดข้อมูล ตรวจความถูกต้อง สรุปจำนวนคลาส และ majority-class baseline |
| 2 | ตัดคำด้วย PyThaiNLP (`engine="longest"`) และเก็บ cache |
| 3 | แปลงเป็นลำดับตัวเลข, padding, แบ่ง train/test 70:30 (`random_state=42`) |
| 4 | ฝึก Word2Vec 3 แบบ (CBOW, Skip-gram, SG-HS) พร้อม **จับเวลา** และสถิติ Huffman tree |
| 5 | สร้างโมเดล CNN, BiLSTM, RNN ตาม Table 7 |
| 6 | ประเมินผลครบทุกด้าน: accuracy, balanced accuracy, precision/recall/F1 ทั้งสองคลาส, macro/weighted F1, MCC, Cohen's κ, ROC-AUC, PR-AUC, confusion matrix, เวลา |
| 7 | วนรันทุกการทดลอง **บันทึกผลทีละรัน** (รันต่อจากจุดค้างได้) |
| 8 | สรุปผลเป็นตารางแบบ Table 8–9, Wilcoxon signed-rank test, เวลาฝึก Word2Vec |
| 9 | กราฟ (300 dpi) และไฟล์ Excel รวมผล |
| 10 | วิเคราะห์รีวิวที่จำแนกผิดของโมเดลที่ดีที่สุด |

**ก่อนรัน:** Runtime → Change runtime type → เลือก **T4 GPU** การทดลองเต็มชุดมี 108 รัน (2 ชุดข้อมูล × 3 embedding × 6 ขนาด × 3 classifier) BiLSTM และ RNN ช้ามากบน CPU ถ้า Colab หลุดกลางทาง ให้รันใหม่ตั้งแต่ต้น ระบบจะข้ามรันที่บันทึกแล้วและทำต่อจากจุดค้าง

**แนะนำ:** ตั้ง `QUICK_TEST = True` รันทดสอบหนึ่งรอบก่อน (ใช้เวลาไม่กี่นาที) แล้วค่อยเปลี่ยนเป็น `False`

## 0. ติดตั้งและตั้งค่า

In [ ]:
!pip -q install pythainlp gensim openpyxl
# ถ้า import ในเซลล์ถัดไป error เรื่อง numpy ให้ Runtime -> Restart session แล้วรันต่อจากเซลล์ถัดไป

In [ ]:
import os, gc, json, time, random, pickle, platform, warnings, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             precision_recall_fscore_support, f1_score,
                             matthews_corrcoef, cohen_kappa_score,
                             roc_auc_score, average_precision_score,
                             confusion_matrix)

import tensorflow as tf
import keras
from keras import layers
from keras.utils import pad_sequences

import gensim
from gensim.models import Word2Vec
import pythainlp
from pythainlp import word_tokenize

warnings.filterwarnings("ignore", category=UserWarning)
print("Python    :", platform.python_version())
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("Gensim    :", gensim.__version__, "| PyThaiNLP:", pythainlp.__version__)
print("GPU       :", tf.config.list_physical_devices("GPU") or "none (CPU only)")

### 0.1 ตั้งค่าการทดลอง (แก้ที่เซลล์นี้เซลล์เดียว)

- ค่าเริ่มต้นตรงกับต้นฉบับ (Table 6–7) ทุกค่า
- `SEEDS = [42]` คือรัน 1 รอบเหมือนต้นฉบับ ถ้าต้องการช่วงความเชื่อมั่นให้ใส่หลายค่า เช่น `[42, 7, 2024, 1, 99]`
- `QUICK_TEST = True` ใช้ทดสอบว่าทุกอย่างทำงาน (ชุดเล็ก, 1 epoch) ก่อนรันจริง
- `EXPERIMENT_NAME` ใช้ตั้งชื่อโฟลเดอร์ผล ถ้ารันซ้ำด้วยชื่อเดิม ระบบจะข้ามรันที่เสร็จแล้ว
- ตัวเลขที่ได้อาจต่างจาก Table 8 เล็กน้อย เพราะการทดลองเดิมไม่ได้ตั้ง seed และ Word2Vec ใช้หลายเธรด
- `DATA_SOURCE = "github"` ดึงข้อมูลจาก GitHub repository นี้ให้อัตโนมัติ ไม่ต้องอัปโหลดไฟล์ ถ้าต้องการใช้ไฟล์ใน Google Drive ของตัวเอง ให้เปลี่ยนเป็น `"drive"` แล้ววางไฟล์ Excel ทั้ง 4 ไฟล์ไว้ที่ `My Drive/dataset_perfume`


In [ ]:
# ---------- Paths ----------
import subprocess
from google.colab import drive
drive.mount("/content/drive")      # results are saved to your Google Drive

# Data source: "github" = clone the public repository (nothing to upload)
#              "drive"  = your own copy of the four Excel files in Google Drive
DATA_SOURCE = "github"
REPO_URL    = "https://github.com/mongkol-saensuk/thai-ecommerce-reviews-sghs"
if DATA_SOURCE == "github":
    REPO_DIR = "/content/thai-ecommerce-reviews-sghs"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, REPO_DIR], check=True)
    DATA_DIR = f"{REPO_DIR}/data"
else:
    DATA_DIR = "/content/drive/My Drive/dataset_perfume"
print("Data folder:", DATA_DIR)

EXPERIMENT_NAME = "sg_hs_methodsx_v1"
RESULTS_DIR = f"/content/drive/My Drive/experiments/{EXPERIMENT_NAME}"

DATASETS = {
    "Perfume": {"reviews": "perfume_review_cleaned_dataset.xlsx",
                "labels":  "perfume_class_cleaned_dataset.xlsx"},
    "Indigo":  {"reviews": "indigo_review_cleaned_dataset.xlsx",
                "labels":  "indigo_class_cleaned_dataset.xlsx"},
}
LABEL_NAMES = {0: "negative", 1: "positive"}

# ---------- Experimental grid (manuscript Tables 6-8) ----------
EMBEDDINGS   = ["CBOW", "SG", "SG-HS"]
VECTOR_SIZES = [50, 100, 150, 200, 250, 300]
CLASSIFIERS  = ["CNN", "BiLSTM", "RNN"]
SEEDS        = [42]

# ---------- Data split ----------
TEST_SIZE    = 0.3
SPLIT_SEED   = 42        # same split as the manuscript
STRATIFY     = False     # manuscript: False; True is recommended for new studies
VAL_FRACTION = 0.0       # 0 = train on the full 70% (manuscript); e.g. 0.1 carves a validation set from it

# ---------- Word2Vec (Table 6) ----------
W2V_WINDOW    = 5
W2V_MIN_COUNT = 1
W2V_EPOCHS    = 5        # Gensim default
W2V_NEGATIVE  = 5        # used by CBOW and SG
W2V_WORKERS   = 3        # Gensim default; set 1 for (near) bit-exact reproducibility

# ---------- Classifiers (Table 7) ----------
EPOCHS        = 10
BATCH_SIZE    = 16
LEARNING_RATE = 1e-3
MAX_LEN       = None     # None = longest review (manuscript)

# ---------- Run control ----------
DETERMINISTIC_OPS = False   # True = tf deterministic ops (slower; some GPU ops may fail)
SAVE_PREDICTIONS  = True
SAVE_W2V_MODELS   = False   # True keeps every Word2Vec model on Drive (uses space)
VERBOSE           = 0       # 0 silent, 1 progress bar, 2 one line per epoch
QUICK_TEST        = False

if QUICK_TEST:
    EXPERIMENT_NAME += "_quicktest"
    RESULTS_DIR = f"/content/drive/My Drive/experiments/{EXPERIMENT_NAME}"
    DATASETS = {"Indigo": DATASETS["Indigo"]}
    VECTOR_SIZES, EPOCHS = [50], 1

for sub in ["", "histories", "predictions", "w2v_models", "figures", "cache"]:
    os.makedirs(os.path.join(RESULTS_DIR, sub), exist_ok=True)

CONFIG = {k: v for k, v in globals().items()
          if k.isupper() and isinstance(v, (int, float, str, bool, list, dict, type(None)))}
with open(f"{RESULTS_DIR}/config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, ensure_ascii=False, indent=2)

n_runs = len(DATASETS) * len(EMBEDDINGS) * len(VECTOR_SIZES) * len(CLASSIFIERS) * len(SEEDS)
print(f"Results folder: {RESULTS_DIR}")
print(f"Planned runs  : {n_runs}  (Word2Vec models: {n_runs // len(CLASSIFIERS)})")

In [ ]:
# Environment record (for the Software line of the Specifications table)
env = {
    "python": platform.python_version(), "tensorflow": tf.__version__,
    "keras": keras.__version__, "gensim": gensim.__version__,
    "pythainlp": pythainlp.__version__, "numpy": np.__version__,
    "pandas": pd.__version__,
    "gpu": [d.name for d in tf.config.list_physical_devices("GPU")],
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
}
import sklearn; env["scikit-learn"] = sklearn.__version__
with open(f"{RESULTS_DIR}/environment.json", "w") as f:
    json.dump(env, f, indent=2)
env

In [ ]:
def set_seed(seed: int):
    # Seed Python, NumPy and TensorFlow so that runs can be repeated.
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        keras.utils.set_random_seed(seed)
    except Exception:
        pass

if DETERMINISTIC_OPS:
    tf.config.experimental.enable_op_determinism()

## 1. โหลดข้อมูลและตรวจความถูกต้อง (Steps 1–2)

In [ ]:
def load_dataset(name):
    cfg = DATASETS[name]
    reviews = pd.read_excel(os.path.join(DATA_DIR, cfg["reviews"]), header=None)[0]
    labels  = pd.read_excel(os.path.join(DATA_DIR, cfg["labels"]),  header=None)[0]
    reviews = reviews.map(str).str.strip()   # same result in pandas 2 and 3: an empty cell becomes "nan"
    assert len(reviews) == len(labels), f"{name}: {len(reviews)} reviews but {len(labels)} labels"
    assert set(labels.unique()) <= {0, 1}, f"{name}: labels must be 0/1, found {sorted(labels.unique())}"
    n_empty = int((reviews == "").sum() + (reviews.str.lower() == "nan").sum())
    if n_empty:
        print(f"WARNING {name}: {n_empty} empty reviews")
    return reviews.tolist(), labels.astype(int).to_numpy()

RAW = {name: load_dataset(name) for name in DATASETS}

rows = []
for name, (texts, y) in RAW.items():
    n, n_pos = len(y), int(y.sum())
    lengths = pd.Series([len(t) for t in texts])
    rows.append({"dataset": name, "n_reviews": n,
                 "n_positive": n_pos, "n_negative": n - n_pos,
                 "pct_positive": round(100 * n_pos / n, 2),
                 "chars_mean": round(lengths.mean(), 1),
                 "chars_median": int(lengths.median()),
                 "chars_max": int(lengths.max())})
dataset_summary = pd.DataFrame(rows)
dataset_summary

## 2. ตัดคำภาษาไทย (Step 3.2)

ใช้ longest matching ของ PyThaiNLP และเก็บผลไว้ใน cache บน Drive เพื่อไม่ต้องตัดคำซ้ำทุกครั้ง

In [ ]:
def tokenize_dataset(name, texts):
    cache = os.path.join(RESULTS_DIR, "cache", f"tokens_{name}.pkl")
    if os.path.exists(cache):
        with open(cache, "rb") as f:
            return pickle.load(f)
    t0 = time.perf_counter()
    tokens = [[w for w in word_tokenize(t, engine="longest") if w.strip()] for t in texts]
    print(f"{name}: tokenised {len(tokens)} reviews in {time.perf_counter() - t0:.1f}s")
    with open(cache, "wb") as f:
        pickle.dump(tokens, f)
    return tokens

TOKENS = {name: tokenize_dataset(name, RAW[name][0]) for name in DATASETS}

for name, toks in TOKENS.items():
    n_tok = pd.Series([len(t) for t in toks])
    vocab = {w for t in toks for w in t}
    dataset_summary.loc[dataset_summary.dataset == name, "tokens_mean"] = round(n_tok.mean(), 1)
    dataset_summary.loc[dataset_summary.dataset == name, "tokens_max"] = int(n_tok.max())
    dataset_summary.loc[dataset_summary.dataset == name, "vocab_size"] = len(vocab)
    print(f"\n{name} — examples:")
    for t in toks[:3]:
        print("  ", " | ".join(t))
dataset_summary

## 3. แปลงเป็นลำดับตัวเลขและแบ่งข้อมูล (Step 5)

สร้างดัชนีคำเรียงตามความถี่ (เหมือน Keras `Tokenizer`), เติม 0 ท้ายลำดับให้ยาวเท่ารีวิวที่ยาวที่สุด แล้วแบ่ง 70:30 ด้วย `random_state=42` ชุดทดสอบจึงเป็นชุดเดียวกับในต้นฉบับ และเป็นชุดเดียวกันสำหรับทุก embedding ทำให้เทียบแบบจับคู่ได้

In [ ]:
def build_vocab(tokens):
    counts = pd.Series([w for t in tokens for w in t]).value_counts(sort=True)
    return {w: i + 1 for i, w in enumerate(counts.index)}   # 0 = padding

def to_sequences(tokens, word_index, max_len=None):
    seqs = [[word_index[w] for w in t if w in word_index] for t in tokens]
    max_len = max_len or max(len(s) for s in seqs)
    return pad_sequences(seqs, maxlen=max_len, padding="post", truncating="post"), max_len

DATA = {}
for name in DATASETS:
    y = RAW[name][1]
    word_index = build_vocab(TOKENS[name])
    X, max_len = to_sequences(TOKENS[name], word_index, MAX_LEN)
    idx = np.arange(len(y))
    idx_tr, idx_te = train_test_split(idx, test_size=TEST_SIZE, random_state=SPLIT_SEED,
                                      stratify=y if STRATIFY else None)
    idx_val = np.array([], dtype=int)
    if VAL_FRACTION > 0:
        idx_tr, idx_val = train_test_split(idx_tr, test_size=VAL_FRACTION,
                                           random_state=SPLIT_SEED, stratify=y[idx_tr])
    majority = max(np.mean(y[idx_te]), 1 - np.mean(y[idx_te]))
    DATA[name] = dict(X=X, y=y, word_index=word_index, max_len=max_len,
                      idx_tr=idx_tr, idx_te=idx_te, idx_val=idx_val,
                      majority_rate=float(majority))
    dataset_summary.loc[dataset_summary.dataset == name, ["n_train", "n_test", "max_len",
                        "test_positive", "test_negative", "majority_rate_test"]] = [
        len(idx_tr), len(idx_te), max_len, int(y[idx_te].sum()),
        int(len(idx_te) - y[idx_te].sum()), round(100 * majority, 2)]
    np.savez(os.path.join(RESULTS_DIR, "cache", f"split_{name}.npz"),
             idx_tr=idx_tr, idx_te=idx_te, idx_val=idx_val)

dataset_summary.to_csv(f"{RESULTS_DIR}/dataset_summary.csv", index=False, encoding="utf-8-sig")
dataset_summary

## 4. ฝึก Word2Vec (Step 4)

| embedding | `sg` | `hs` | `negative` |
|:--|:--:|:--:|:--:|
| CBOW | 0 | 0 | 5 |
| SG (Skip-gram + negative sampling) | 1 | 0 | 5 |
| SG-HS (Skip-gram + hierarchical softmax) | 1 | 1 | 0 |

บันทึก **เวลาฝึก** ของแต่ละโมเดล (ใช้ตอบข้อจำกัดเรื่องเวลาในต้นฉบับ) และสำหรับ SG-HS บันทึกความยาว Huffman code เฉลี่ยแบบถ่วงความถี่ เทียบกับ log₂V

In [ ]:
W2V_LOG = os.path.join(RESULTS_DIR, "w2v_log.csv")
W2V_COLS = ["dataset", "embedding", "vector_size", "seed", "w2v_train_sec", "w2v_vocab",
            "w2v_sec_per_epoch", "huff_mean_code_len", "huff_weighted_code_len",
            "huff_max_code_len", "log2_V"]

def w2v_params(kind, d, seed):
    p = dict(vector_size=d, window=W2V_WINDOW, min_count=W2V_MIN_COUNT,
             epochs=W2V_EPOCHS, workers=W2V_WORKERS, seed=seed)
    if kind == "CBOW":
        p.update(sg=0, hs=0, negative=W2V_NEGATIVE)
    elif kind == "SG":
        p.update(sg=1, hs=0, negative=W2V_NEGATIVE)
    elif kind == "SG-HS":
        p.update(sg=1, hs=1, negative=0)
    else:
        raise ValueError(kind)
    return p

def huffman_stats(model):
    # Frequency-weighted and plain mean Huffman code length (hierarchical softmax only).
    wv = model.wv
    try:
        lengths = np.array([len(wv.get_vecattr(w, "code")) for w in wv.index_to_key])
        counts = np.array([wv.get_vecattr(w, "count") for w in wv.index_to_key])
    except Exception:
        return {}
    V = len(wv.index_to_key)
    return {"huff_mean_code_len": float(lengths.mean()),
            "huff_weighted_code_len": float((lengths * counts).sum() / counts.sum()),
            "huff_max_code_len": int(lengths.max()),
            "log2_V": float(np.log2(V))}

def train_w2v(dataset, kind, d, seed):
    set_seed(seed)
    params = w2v_params(kind, d, seed)
    t0 = time.perf_counter()
    model = Word2Vec(sentences=TOKENS[dataset], **params)
    elapsed = time.perf_counter() - t0
    info = {"dataset": dataset, "embedding": kind, "vector_size": d, "seed": seed,
            "w2v_train_sec": round(elapsed, 3), "w2v_vocab": len(model.wv.index_to_key),
            "w2v_sec_per_epoch": round(elapsed / W2V_EPOCHS, 4)}
    if params["hs"]:
        info.update(huffman_stats(model))
    pd.DataFrame([info]).reindex(columns=W2V_COLS).to_csv(W2V_LOG, mode="a", index=False,
                                header=not os.path.exists(W2V_LOG), encoding="utf-8-sig")
    if SAVE_W2V_MODELS:
        model.save(os.path.join(RESULTS_DIR, "w2v_models", f"{dataset}_{kind}_d{d}_s{seed}.model"))
    return model, info

def embedding_matrix(model, word_index):
    E = np.zeros((len(word_index) + 1, model.vector_size), dtype="float32")
    hits = 0
    for w, i in word_index.items():
        if w in model.wv:
            E[i] = model.wv[w]; hits += 1
    return E, hits / len(word_index)

## 5. โมเดลจำแนก (Step 5, Table 7)

ชั้น embedding ถูกตั้งค่าด้วยเวกเตอร์ Word2Vec และ **ไม่ฝึกต่อ** (`trainable=False`) โมเดลจึงเห็นเวกเตอร์จาก Step 4 ตรง ๆ

In [ ]:
def build_model(kind, E, max_len):
    inp = keras.Input(shape=(max_len,), dtype="int32")
    emb = layers.Embedding(E.shape[0], E.shape[1], trainable=False, name="w2v_embedding")
    x = emb(inp)
    if kind == "CNN":
        x = layers.Conv1D(128, 3, activation="relu")(x)
        x = layers.GlobalMaxPooling1D()(x)
    elif kind == "BiLSTM":
        x = layers.Bidirectional(layers.LSTM(128))(x)
        x = layers.Dropout(0.5)(x)
        x = layers.Dense(64, activation="relu")(x)
    elif kind == "RNN":
        x = layers.SimpleRNN(128)(x)
        x = layers.Dropout(0.5)(x)
    else:
        raise ValueError(kind)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inp, out, name=kind)
    emb.set_weights([E])                      # load the Word2Vec vectors
    model.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE),
                  loss="binary_crossentropy", metrics=["accuracy"])
    return model

## 6. ตัวชี้วัด (Step 6)

คลาสบวก (1) เป็น positive class ในการคำนวณ precision/recall/F1 ของคลาสบวก และคำนวณของคลาสลบแยกด้วย เพราะชุดครามไม่สมดุล (บวก 76%)
- `macro_f1` ให้น้ำหนักสองคลาสเท่ากัน เป็นตัวชี้วัดหลักที่ควรรายงานคู่กับ accuracy
- `mcc` (Matthews correlation) ใช้ได้ดีเมื่อคลาสไม่สมดุล โมเดลที่ทายคลาสเดียวทั้งหมดจะได้ 0
- `collapsed = True` แปลว่าโมเดลทายคลาสเดียวทั้งชุดทดสอบ

In [ ]:
def evaluate(y_true, prob, threshold=0.5):
    y_pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    both = len(np.unique(y_true)) == 2
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_pos": p[1], "recall_pos": r[1], "f1_pos": f[1],
        "precision_neg": p[0], "recall_neg": r[0], "f1_neg": f[0],
        "macro_precision": p.mean(), "macro_recall": r.mean(),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "mcc": matthews_corrcoef(y_true, y_pred) if len(np.unique(y_pred)) > 1 else 0.0,
        "cohen_kappa": cohen_kappa_score(y_true, y_pred) if len(np.unique(y_pred)) > 1 else 0.0,
        "roc_auc": roc_auc_score(y_true, prob) if both else np.nan,
        "pr_auc_pos": average_precision_score(y_true, prob) if both else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "pred_pos_rate": float(y_pred.mean()),
        "collapsed": bool(len(np.unique(y_pred)) == 1),
    }

## 7. รันการทดลองทั้งหมด

- ฝึก Word2Vec **ครั้งเดียว** ต่อ (ชุดข้อมูล, embedding, ขนาดเวกเตอร์, seed) แล้วใช้ร่วมกันทั้ง 3 classifier
- ทุกรันบันทึกลง `results_runs.csv` ทันที ถ้า Colab หลุด ให้รันเซลล์ข้างบนใหม่แล้วรันเซลล์นี้อีกครั้ง ระบบจะข้ามรันที่เสร็จแล้ว
- เก็บ history การฝึกของทุกรัน (`histories/`) และความน่าจะเป็นที่ทายบนชุดทดสอบ (`predictions/`)

In [ ]:
RUNS_CSV = os.path.join(RESULTS_DIR, "results_runs.csv")
KEY = ["dataset", "embedding", "vector_size", "classifier", "seed"]

def done_keys():
    if not os.path.exists(RUNS_CSV):
        return set()
    df = pd.read_csv(RUNS_CSV)
    return set(map(tuple, df[KEY].astype(str).values))

def run_id(ds, emb, d, clf, seed):
    return f"{ds}_{emb}_d{d}_{clf}_s{seed}"

def run_all():
    finished = done_keys()
    total = len(DATASETS) * len(EMBEDDINGS) * len(VECTOR_SIZES) * len(CLASSIFIERS) * len(SEEDS)
    print(f"{len(finished)} of {total} runs already done")
    t_start = time.perf_counter()
    for ds in DATASETS:
        D = DATA[ds]
        X_tr, y_tr = D["X"][D["idx_tr"]], D["y"][D["idx_tr"]]
        X_te, y_te = D["X"][D["idx_te"]], D["y"][D["idx_te"]]
        val = (D["X"][D["idx_val"]], D["y"][D["idx_val"]]) if len(D["idx_val"]) else None
        for seed in SEEDS:
            for emb in EMBEDDINGS:
                for d in VECTOR_SIZES:
                    todo = [c for c in CLASSIFIERS
                            if tuple(map(str, (ds, emb, d, c, seed))) not in finished]
                    if not todo:
                        continue
                    w2v, w2v_info = train_w2v(ds, emb, d, seed)
                    E, coverage = embedding_matrix(w2v, D["word_index"])
                    del w2v
                    for clf in todo:
                        rid = run_id(ds, emb, d, clf, seed)
                        keras.backend.clear_session(); gc.collect()
                        set_seed(seed)
                        model = build_model(clf, E, D["max_len"])
                        t0 = time.perf_counter()
                        hist = model.fit(X_tr, y_tr, epochs=EPOCHS, batch_size=BATCH_SIZE,
                                         validation_data=val, verbose=VERBOSE)
                        train_sec = time.perf_counter() - t0
                        t0 = time.perf_counter()
                        prob = model.predict(X_te, batch_size=256, verbose=0).ravel()
                        infer_sec = time.perf_counter() - t0
                        m = evaluate(y_te, prob)
                        row = {"run_id": rid, "dataset": ds, "embedding": emb, "vector_size": d,
                               "classifier": clf, "seed": seed, **m,
                               "majority_rate": D["majority_rate"],
                               "acc_minus_majority": m["accuracy"] - D["majority_rate"],
                               "train_sec": round(train_sec, 2), "infer_sec": round(infer_sec, 3),
                               "w2v_train_sec": w2v_info["w2v_train_sec"],
                               "w2v_vocab": w2v_info["w2v_vocab"],
                               "emb_coverage": round(coverage, 4),
                               "n_params": model.count_params(),
                               "final_train_loss": float(hist.history["loss"][-1]),
                               "final_train_acc": float(hist.history["accuracy"][-1]),
                               "timestamp": datetime.datetime.now().isoformat(timespec="seconds")}
                        pd.DataFrame([row]).to_csv(RUNS_CSV, mode="a", index=False,
                                                   header=not os.path.exists(RUNS_CSV),
                                                   encoding="utf-8-sig")
                        with open(os.path.join(RESULTS_DIR, "histories", rid + ".json"), "w") as f:
                            json.dump({k: [float(v) for v in vals]
                                       for k, vals in hist.history.items()}, f)
                        if SAVE_PREDICTIONS:
                            np.save(os.path.join(RESULTS_DIR, "predictions", rid + ".npy"),
                                    prob.astype("float32"))
                        finished.add(tuple(map(str, (ds, emb, d, clf, seed))))
                        elapsed = (time.perf_counter() - t_start) / 60
                        print(f"[{len(finished):>3}/{total}] {rid:<32} "
                              f"acc={m['accuracy']*100:6.2f}  macroF1={m['macro_f1']*100:6.2f}  "
                              f"MCC={m['mcc']:.3f}  train={train_sec:6.1f}s  "
                              f"{'COLLAPSED ' if m['collapsed'] else ''}({elapsed:.1f} min)")
                        del model
    print("All runs finished.")

run_all()

## 8. สรุปผล

ถ้ารันหลาย seed ตารางจะใช้ค่าเฉลี่ยข้าม seed ก่อน แล้วจึงสรุปข้ามขนาดเวกเตอร์

In [ ]:
runs = pd.read_csv(RUNS_CSV).drop_duplicates(subset=KEY, keep="last")
EMB_ORDER = {e: i for i, e in enumerate(EMBEDDINGS)}
DS_ORDER  = {d: i for i, d in enumerate(DATASETS)}
CLF_ORDER = {c: i for i, c in enumerate(CLASSIFIERS)}
METRICS = ["accuracy", "macro_f1", "balanced_accuracy", "mcc", "f1_pos", "f1_neg",
           "roc_auc", "pr_auc_pos", "cohen_kappa", "train_sec", "w2v_train_sec"]

def sort_frame(df):
    order = {"dataset": DS_ORDER, "embedding": EMB_ORDER, "classifier": CLF_ORDER}
    return df.sort_values(["dataset", "embedding", "classifier"],
                          key=lambda s: s.map(order[s.name]))

# mean over seeds for each (dataset, embedding, classifier, vector size)
by_cfg = (runs.groupby(["dataset", "embedding", "classifier", "vector_size"])[METRICS]
              .mean().reset_index())

def wide_table(metric, scale=100, digits=2):
    t = by_cfg.pivot_table(index=["dataset", "embedding", "classifier"],
                           columns="vector_size", values=metric)
    t = (t * scale).round(digits).reset_index()
    t.columns.name = None
    return sort_frame(t).reset_index(drop=True)

table_accuracy = wide_table("accuracy")     # same layout as manuscript Table 8
table_macro_f1 = wide_table("macro_f1")
table_mcc      = wide_table("mcc", scale=1, digits=3)

def across_sizes(metric, scale=100):
    g = by_cfg.groupby(["dataset", "classifier", "embedding"])
    out = g[metric].agg(["mean", "std", "max"]).mul(scale).round(2)
    out["best_at_d"] = g.apply(lambda x: ", ".join(
        str(v) for v in x.loc[np.isclose(x[metric], x[metric].max()), "vector_size"]))
    out.columns = [f"{metric}_mean", f"{metric}_sd", f"{metric}_best", f"{metric}_best_at_d"]
    return out

table_summary = (pd.concat([across_sizes("accuracy"), across_sizes("macro_f1"),
                            across_sizes("mcc", scale=1)], axis=1).reset_index())
table_summary = table_summary.sort_values(
    ["dataset", "classifier", "embedding"],
    key=lambda s: s.map({"dataset": DS_ORDER, "embedding": EMB_ORDER,
                         "classifier": CLF_ORDER}[s.name])).reset_index(drop=True)

print("Accuracy (%) — Table 8 layout")
display(table_accuracy)
print("Macro-F1 (%)")
display(table_macro_f1)
print("Summary across vector sizes — Table 9 layout")
display(table_summary)

In [ ]:
# Wilcoxon signed-rank test: SG-HS vs each baseline, paired by dataset x classifier x vector size x seed
def paired_test(metric, baseline, datasets=None, classifiers=None):
    keys = ["dataset", "classifier", "vector_size", "seed"]
    a = runs[runs.embedding == "SG-HS"].set_index(keys)[metric]
    b = runs[runs.embedding == baseline].set_index(keys)[metric]
    pair = pd.concat([a, b], axis=1, keys=["sghs", "base"]).dropna()
    if datasets:
        pair = pair[pair.index.get_level_values("dataset").isin(datasets)]
    if classifiers:
        pair = pair[pair.index.get_level_values("classifier").isin(classifiers)]
    diff = pair["sghs"] - pair["base"]
    res = {"metric": metric, "baseline": baseline,
           "datasets": "+".join(datasets) if datasets else "all",
           "classifiers": "+".join(classifiers) if classifiers else "all", "n_pairs": len(diff),
           "sghs_wins": int((diff > 0).sum()), "ties": int((diff == 0).sum()),
           "mean_diff_pts": round(100 * diff.mean(), 2) if metric != "mcc" else round(diff.mean(), 4),
           "sd_diff_pts": round(100 * diff.std(ddof=1), 2) if metric != "mcc" else round(diff.std(ddof=1), 4)}
    if len(diff) >= 6 and (diff != 0).any():
        w = stats.wilcoxon(pair["sghs"], pair["base"])
        res.update({"W": float(w.statistic), "p_value": float(w.pvalue)})
    return res

# scope "CNN+BiLSTM" matches the manuscript analysis; "all" also includes the RNN
tests = []
for metric in ["accuracy", "macro_f1", "mcc"]:
    for base in ["SG", "CBOW"]:
        for clfs in (["CNN", "BiLSTM"], None):
            tests.append(paired_test(metric, base, None, clfs))
            for ds in DATASETS:
                tests.append(paired_test(metric, base, [ds], clfs))
table_wilcoxon = pd.DataFrame(tests)
table_wilcoxon

In [ ]:
# Word2Vec training time and Huffman statistics
w2v_log = pd.read_csv(W2V_LOG).drop_duplicates(
    subset=["dataset", "embedding", "vector_size", "seed"], keep="last")
table_w2v_time = (w2v_log.pivot_table(index=["dataset", "embedding"], columns="vector_size",
                                      values="w2v_train_sec", aggfunc="mean")
                         .round(2).reset_index())
table_w2v_time.columns.name = None
table_w2v_time = table_w2v_time.sort_values(
    ["dataset", "embedding"],
    key=lambda s: s.map({"dataset": DS_ORDER, "embedding": EMB_ORDER}[s.name])).reset_index(drop=True)
huff_cols = ["huff_mean_code_len", "huff_weighted_code_len", "huff_max_code_len", "log2_V"]
table_huffman = (w2v_log[w2v_log.embedding == "SG-HS"]
                 .groupby("dataset")[huff_cols + ["w2v_vocab"]].mean().round(2).reset_index())

print("Word2Vec training time (s)")
display(table_w2v_time)
print("Huffman tree statistics (SG-HS)")
display(table_huffman)

## 9. กราฟและไฟล์รวมผล

กราฟทั้งหมดบันทึกเป็น PNG 300 dpi ในโฟลเดอร์ `figures/` และตารางทั้งหมดรวมใน `results_summary.xlsx`

In [ ]:
COLORS = {"CBOW": "#1baf7a", "SG": "#eb6834", "SG-HS": "#2a78d6"}
MARKERS = {"CBOW": "^", "SG": "s", "SG-HS": "o"}
LABELS = {"CBOW": "CBOW", "SG": "Skip-gram (NS)", "SG-HS": "Skip-gram + HS"}
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

def plot_metric_grid(metric, ylabel, fname, scale=100):
    ds_list, clf_list = list(DATASETS), CLASSIFIERS
    fig, axes = plt.subplots(len(ds_list), len(clf_list),
                             figsize=(3.4 * len(clf_list), 2.9 * len(ds_list)),
                             squeeze=False, sharex=True)
    for i, ds in enumerate(ds_list):
        for j, clf in enumerate(clf_list):
            ax = axes[i, j]
            for emb in EMBEDDINGS:
                sub = by_cfg[(by_cfg.dataset == ds) & (by_cfg.classifier == clf) &
                             (by_cfg.embedding == emb)].sort_values("vector_size")
                if len(sub):
                    ax.plot(sub.vector_size, sub[metric] * scale, color=COLORS[emb],
                            marker=MARKERS[emb], lw=2, ms=6, mec="white", label=LABELS[emb])
            if metric == "accuracy":
                ax.axhline(DATA[ds]["majority_rate"] * 100, color="#8a8984", ls="--", lw=1)
            ax.set_title(f"{ds} — {clf}", fontsize=10, loc="left")
            ax.set_xticks(VECTOR_SIZES); ax.grid(axis="y", color="#e6e5e0")
            if j == 0: ax.set_ylabel(ylabel)
            if i == len(ds_list) - 1: ax.set_xlabel("Vector size")
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=3, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 0.94))
    fig.savefig(os.path.join(RESULTS_DIR, "figures", fname), dpi=300, facecolor="white")
    plt.show()

plot_metric_grid("accuracy", "Accuracy (%)", "accuracy_vs_vector_size.png")
plot_metric_grid("macro_f1", "Macro-F1 (%)", "macro_f1_vs_vector_size.png")

In [ ]:
# Word2Vec training time by embedding
fig, axes = plt.subplots(1, len(DATASETS), figsize=(5 * len(DATASETS), 3.2), squeeze=False)
for k, ds in enumerate(DATASETS):
    ax = axes[0, k]
    for emb in EMBEDDINGS:
        sub = (w2v_log[(w2v_log.dataset == ds) & (w2v_log.embedding == emb)]
               .groupby("vector_size").w2v_train_sec.mean())
        ax.plot(sub.index, sub.values, color=COLORS[emb], marker=MARKERS[emb], lw=2,
                ms=6, mec="white", label=LABELS[emb])
    ax.set_title(ds, loc="left"); ax.set_xlabel("Vector size"); ax.set_xticks(VECTOR_SIZES)
    ax.grid(axis="y", color="#e6e5e0")
axes[0, 0].set_ylabel("Word2Vec training time (s)")
axes[0, 0].legend(frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "figures", "w2v_training_time.png"), dpi=300, facecolor="white")
plt.show()

# Confusion matrix of the best run (by macro-F1) for each dataset
best_runs = runs.loc[runs.groupby("dataset").macro_f1.idxmax()]
fig, axes = plt.subplots(1, len(best_runs), figsize=(4 * len(best_runs), 3.6), squeeze=False)
for k, (_, r) in enumerate(best_runs.iterrows()):
    cm = np.array([[r.tn, r.fp], [r.fn, r.tp]])
    ax = axes[0, k]
    ax.imshow(cm, cmap="Blues")
    for (a, b), v in np.ndenumerate(cm):
        ax.text(b, a, f"{v}\n({v / cm[a].sum() * 100:.1f}%)", ha="center", va="center",
                color="white" if v > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], ["neg", "pos"]); ax.set_yticks([0, 1], ["neg", "pos"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"{r.dataset}: {r.embedding}+{r.classifier}, d={r.vector_size}\n"
                 f"acc {r.accuracy*100:.2f}%, macro-F1 {r.macro_f1*100:.2f}%", fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "figures", "confusion_best.png"), dpi=300, facecolor="white")
plt.show()
best_runs[["run_id", "accuracy", "macro_f1", "mcc", "f1_neg", "f1_pos", "roc_auc"]]

In [ ]:
# All tables in one Excel workbook
xlsx = os.path.join(RESULTS_DIR, "results_summary.xlsx")
with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
    dataset_summary.to_excel(xw, sheet_name="dataset_summary", index=False)
    table_accuracy.to_excel(xw, sheet_name="T8_accuracy", index=False)
    table_macro_f1.to_excel(xw, sheet_name="macro_f1", index=False)
    table_mcc.to_excel(xw, sheet_name="mcc", index=False)
    table_summary.to_excel(xw, sheet_name="T9_summary", index=False)
    table_wilcoxon.to_excel(xw, sheet_name="wilcoxon", index=False)
    table_w2v_time.to_excel(xw, sheet_name="w2v_time", index=False)
    if len(table_huffman):
        table_huffman.to_excel(xw, sheet_name="huffman", index=False)
    runs.to_excel(xw, sheet_name="all_runs", index=False)
    pd.DataFrame(list(CONFIG.items()), columns=["setting", "value"]).astype(str) \
      .to_excel(xw, sheet_name="config", index=False)
print("Saved:", xlsx)
print(sorted(os.listdir(RESULTS_DIR)))

## 10. วิเคราะห์รีวิวที่จำแนกผิด

ใช้ความน่าจะเป็นที่บันทึกไว้ของโมเดลที่ดีที่สุดในแต่ละชุดข้อมูล (ตาม macro-F1) บันทึกเป็น `misclassified_<dataset>.csv` เรียงจากที่โมเดลมั่นใจผิดมากที่สุด

In [ ]:
for _, r in best_runs.iterrows():
    path = os.path.join(RESULTS_DIR, "predictions", r.run_id + ".npy")
    if not os.path.exists(path):
        continue
    D = DATA[r.dataset]
    prob = np.load(path)
    texts = np.array(RAW[r.dataset][0], dtype=object)[D["idx_te"]]
    toks = [" | ".join(TOKENS[r.dataset][i]) for i in D["idx_te"]]
    y_true = D["y"][D["idx_te"]]
    y_pred = (prob >= 0.5).astype(int)
    err = pd.DataFrame({"review": texts, "tokens": toks, "true": y_true,
                        "pred": y_pred, "prob_positive": prob.round(4)})
    err = err[err.true != err.pred].copy()
    err["confidence_wrong"] = np.where(err.pred == 1, err.prob_positive, 1 - err.prob_positive)
    err = err.sort_values("confidence_wrong", ascending=False)
    err["true"] = err["true"].map(LABEL_NAMES); err["pred"] = err["pred"].map(LABEL_NAMES)
    out = os.path.join(RESULTS_DIR, f"misclassified_{r.dataset}.csv")
    err.to_csv(out, index=False, encoding="utf-8-sig")
    print(f"{r.dataset}: {len(err)} errors out of {len(y_true)} -> {out}")
    display(err.head(10))

### ไฟล์ที่ได้ใน `RESULTS_DIR`

| ไฟล์ | เนื้อหา |
|:--|:--|
| `results_runs.csv` | ทุกตัวชี้วัดของทุกรัน (1 แถว/รัน) |
| `results_summary.xlsx` | ตารางแบบ Table 8–9, macro-F1, MCC, Wilcoxon, เวลา Word2Vec, Huffman, config |
| `w2v_log.csv` | เวลาฝึกและขนาดคำศัพท์ของ Word2Vec ทุกโมเดล |
| `dataset_summary.csv` | จำนวนรีวิว คลาส ความยาว และ majority-class rate |
| `histories/*.json` | loss/accuracy รายรอบของทุกรัน |
| `predictions/*.npy` | ความน่าจะเป็นที่ทายบนชุดทดสอบของทุกรัน |
| `figures/*.png` | กราฟ 300 dpi |
| `misclassified_*.csv` | รีวิวที่ทายผิดของโมเดลที่ดีที่สุด |
| `config.json`, `environment.json` | ค่าที่ตั้งและเวอร์ชันซอฟต์แวร์ |